In [1]:
!pip install selenium webdriver-manager supabase

import os
import re
import time
import logging
from dataclasses import dataclass
from typing import Optional

#from dotenv import load_dotenv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from supabase import create_client, Client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

In [3]:
from google.colab import userdata

#This is for colab, using Colab secrets
SUPABASE_URL = "https://xnpdjsmadwyyxjapwxre.supabase.co"
SUPABASE_KEY = "sb_publishable_b3klFBb9OkaT4qXTMe-ljQ_2BIpLB-o"

SETS_TABLE:   str = "tcg_sets"
PRICES_TABLE: str = "tcg_card_prices"

#Choosing sets to scrape

SETS_TO_SCRAPE = [
        {
        "set_name": "Premium Booster -The Best- Vol. 2",
        "set_code": "PRB02",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/premium-booster-the-best-vol-2",
    },
    {
        "set_name": "Legacy of the Master",
        "set_code": "OP12",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/legacy-of-the-master",
    },
    {
        "set_name": "A Fist of Divine Speed",
        "set_code": "OP11",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/a-fist-of-divine-speed",
    },
    {
        "set_name": "Royal Blood",
        "set_code": "OP10",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/royal-blood",
    },
    {
        "set_name": "Emperors in the New World",
        "set_code": "OP09",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/emperors-in-the-new-world",
    },
    {
        "set_name": "Carrying On His Will",
        "set_code": "OP13",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/carrying-on-his-will",
    },
    {
        "set_name": "Extra Booster: Anime 25th Collection Guide",
        "set_code": "EB02",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/extra-booster-anime-25th-collection",
    },

    {
        "set_name": "Extra Booster: One Piece Heroines Edition",
        "set_code": "EB03",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/extra-booster-one-piece-heroines-edition",
    },
    {
        "set_name": "The Azure Seas Seven",
        "set_code": "OP14",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/the-azure-seas-seven"
    },
    {
        "set_name": "Adventure on Kami's Island",
        "set_code": "OP15",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/adventure-on-kamis-island"
    },
    {
        "set_name": "The Time of Battle",
        "set_code": "OP16",
        "url": "https://www.tcgplayer.com/categories/trading-and-collectible-card-games/one-piece-card-game/price-guides/the-time-of-battle"
        }
]

In [4]:
#Parameters for selenium
PAGE_LOAD_TIMEOUT = 60 # Changed from 700 to 60
SCROLL_PAUSE      = 2
BATCH_SIZE        = 500

#The actual parseable data. THis will be converted to match the values in the supabase table
@dataclass
class SetRecord:
    set_name: str
    set_code: str
    url:      str


@dataclass
class CardPrice:
    set_code:     str
    product_name: str
    card_number:  Optional[str]
    printing:     Optional[str]
    condition:    Optional[str]
    rarity:       Optional[str]
    market_price: Optional[float]

In [5]:
#Download the version of chrome that is compatible with selenium and colab

!pip install selenium webdriver-manager -q
!apt-get install -y wget unzip -q
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q

Reading package lists...
Building dependency tree...
Reading state information...
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newest version (1.21.2-2ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 3 not upgraded.
Need to get 11.2 MB/141 MB of archives.
After this operation, 474 MB of additional disk space will be used.
Get:1 http://a

In [6]:
log.info("Attempting to fix Chrome installation...")
!apt-get update -q

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [97.8 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,258 kB]


In [7]:
# Re-install Chrome after updating apt-get
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q
log.info("Chrome installation re-attempted.")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 62 not upgraded.
Need to get 11.2 MB/141 MB of archives.
After this operation, 474 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatspi2.0-0 

In [8]:
# Confirm Chrome installation
import subprocess
print(subprocess.getoutput("which google-chrome"))
print(subprocess.getoutput("google-chrome --version"))

/usr/bin/google-chrome
Google Chrome 149.0.7827.200 


In [9]:
# Re-running the main function now that Chrome is installed
#main()

In [10]:
#Tells which version of Chrome is being used
import subprocess
print(subprocess.getoutput("which google-chrome"))
print(subprocess.getoutput("google-chrome --version"))


/usr/bin/google-chrome
Google Chrome 149.0.7827.200 


In [11]:
# Configures Headless chrome driver for scraping
def build_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    # Fix 1: match UA to actual installed Chrome version
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/149.0.0.0 Safari/537.36"
    )
    # Fix 3: suppress navigator.webdriver flag to avoid bot detection
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    # Patch: remove webdriver property at the JS level
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


    #Scrolls the given WebDriver instance to the bottom of the page
    #Waits for a 10 second pause between scrolls until no new content loads.

def scroll_to_bottom(driver: webdriver.Chrome) -> None:
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
        log.info("  Scrolled → page height %d px", new_height)


    #Parses a raw string, extracting numeric price data and returning it as a float.
    #Returns None if parsing fails or the string is empty/non-numeric.
def parse_price(raw: str) -> Optional[float]:
    clean = re.sub(r"[^\d.]", "", raw)
    try:
        return float(clean) if clean else None
    except ValueError:
        return None

    # Retrieves the stripped text content from a specific cell by index.
    # Returns None if the index is out of bounds or the cell's text is empty.

def cell_text(cells: list, index: int) -> Optional[str]:
    if index < len(cells):
        val = cells[index].text.strip()
        return val if val else None
    return None


In [ ]:
# Does the Scraping

def scrape_set(set_record: SetRecord) -> list[CardPrice]:
    """
    Scrape all card rows from one TCGPlayer price guide page.

    TCGPlayer table column order (from screenshot):
        0: checkbox  1: image  2: product name  3: printing
        4: condition  5: rarity  6: card number  7: market price
    """
    driver = build_driver()
    results: list[CardPrice] = []

    try:
        log.info("Scraping: %s", set_record.set_name)
        driver.get(set_record.url)

        # Fix 2: give JS time to bootstrap, then log diagnostics so
        # you can see whether it's a bot block or a selector mismatch
        time.sleep(5)
        log.info("  Page title: %s", driver.title)
        log.info("  Page source snippet:\n%s", driver.page_source[:2000])

        # Wait for the main body element to be visible
        WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(
            EC.visibility_of_element_located(
                (By.CSS_SELECTOR, "tbody.tcg-table-body tr")
            )
        )
        time.sleep(3)
        scroll_to_bottom(driver)
        time.sleep(2)

        rows = driver.find_elements(
            By.CSS_SELECTOR, "tbody.tcg-table-body tr"
        )
        log.info("  Found %d candidate rows", len(rows))

        for row in rows:
            try:
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) < 6:
                    continue

                product_name = cell_text(cells, 2)
                if not product_name:
                    continue

                results.append(
                    CardPrice(
                        set_code=set_record.set_code,
                        product_name=product_name,
                        printing=cell_text(cells, 3),
                        condition=cell_text(cells, 4),
                        rarity=cell_text(cells, 5),
                        card_number=cell_text(cells, 6),
                        market_price=parse_price(cell_text(cells, 7) or ""),
                    )
                )
            except Exception as e:
                log.warning("  Skipping row: %s", e)

    finally:
        driver.quit()

    log.info("  Scraped %d card entries", len(results))
    return results


In [13]:
#Helper functions

def get_client() -> Client:
    if not SUPABASE_URL or not SUPABASE_KEY:
        raise ValueError("Set SUPABASE_URL and SUPABASE_KEY in your .env file.")
    return create_client(SUPABASE_URL, SUPABASE_KEY)

    """
    Insert or update the set row and return set_code.
    set_code is the unique conflict key. This way re-runs never create duplicate sets.
    """

def upsert_set(client: Client, set_record: SetRecord) -> str:

    client.table(SETS_TABLE).upsert(
        {
            "set_name": set_record.set_name,
            "set_code": set_record.set_code,
            "url":      set_record.url,
        },
        on_conflict="set_code",
    ).execute()
    log.info("  Set '%s' → set_code='%s'", set_record.set_name, set_record.set_code)
    return set_record.set_code


def insert_prices(client: Client, cards: list[CardPrice]) -> None:
    if not cards:
        log.warning("  No card prices to insert.")
        return

    records = [
        {
            "set_code":     c.set_code,
            "product_name": c.product_name,
            "card_number":  c.card_number,
            "printing":     c.printing,
            "condition":    c.condition,
            "rarity":       c.rarity,
            "market_price": c.market_price,
        }
        for c in cards
    ]

    for i in range(0, len(records), BATCH_SIZE):
        chunk = records[i : i + BATCH_SIZE]
        client.table(PRICES_TABLE).insert(chunk).execute()
        log.info("  Inserted rows %d–%d", i + 1, i + len(chunk))

    log.info("%d rows inserted into '%s'", len(records), PRICES_TABLE)


In [14]:
#This method actually puts the stuff into the database and into a CSV file

def main() -> None:

    for set_cfg in SETS_TO_SCRAPE:
        set_record = SetRecord(**set_cfg)

        client = get_client()

        # 1. Ensure the set exists in tcg_sets
        upsert_set(client, set_record) #Comment out if you just want the CSV

        # 2. Scrape the price guide
        cards = scrape_set(set_record)
        if not cards:
            log.error("  No data scraped")
            continue

        # Convert to DataFrame and save to CSV
        card_data = [c.__dict__ for c in cards]
        df = pd.DataFrame(card_data)
        csv_filename = f"{set_record.set_code}.csv"
        df.to_csv(csv_filename, index=False)
        log.info("  Saved %d card entries to %s", len(df), csv_filename)

        # Preview first 3 rows
        log.info("  Sample rows:")
        for c in cards[:3]:
            log.info("    %s", c)

        # 4. Append to Supabase
        insert_prices(client, cards) #Comment out if you just want the CSV

        time.sleep(7)

    log.info("All sets complete.")


if __name__ == "__main__":
    main()

In [ ]:
# Re-running the main function now that Chrome is installed
main()

In [ ]:
"""
## SQL USED TO CREATE THE DATABASE TABLES

create table tcg_sets(
  id bigserial primary key,
  set_name text not null,
  set_code text not null,
  url text,
  created_at timestamptz default now(),
  unique(set_code));

create table tcg_card_prices(
  id bigserial primary key,
  set_code text not null references tcg_sets(set_code),
  product_name text not null,
  card_number text,
  printing text,
  condition text,
  rarity text,
  market_price numeric(10,2),
  scraped_at timestamptz default now());
  """